In [1]:
import pandas as pd
import duckdb
import hashlib
import numpy as np

In [2]:
# Step 1. Raw 데이터 로드
# raw_sample_1000_users.csv 파일을 읽어 raw_df에 저장하세요.
# 이후 데이터 구조를 확인하기 위해 head(), shape, columns도 출력하세요.
raw_df = pd.read_csv("../data/raw_sample_1000_users.csv")

print(raw_df.head())
print(raw_df.shape)
print(raw_df.columns.tolist())

  InvoiceNo StockCode                         Description  Quantity  \
0    536381     22139    RETROSPOT TEA SET CERAMIC 11 PC         23   
1    536381     84854                 GIRLY PINK TOOL SET         5   
2    536381     22411   JUMBO SHOPPER VINTAGE RED PAISLEY        10   
3    536381     82567           AIRLINE LOUNGE,METAL SIGN         2   
4    536381     21672  WHITE SPOT RED CERAMIC DRAWER KNOB         6   

           InvoiceDate  UnitPrice  CustomerID         Country  revenue  
0  2010-12-01 09:41:00       4.25       15311  United Kingdom    97.75  
1  2010-12-01 09:41:00       4.95       15311  United Kingdom    24.75  
2  2010-12-01 09:41:00       1.95       15311  United Kingdom    19.50  
3  2010-12-01 09:41:00       2.10       15311  United Kingdom     4.20  
4  2010-12-01 09:41:00       1.25       15311  United Kingdom     7.50  
(372340, 9)
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'revenue']


C:\Users\SSAFY\AppData\Local\Temp\ipykernel_29084\2461730276.py:4: DtypeWarning: Columns (0: InvoiceNo) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv("../data/raw_sample_1000_users.csv")


In [4]:
# Step 2. 이벤트 테이블 생성
# 아래 컬럼명을 분석용 이름으로 변경합니다.
# CustomerID -> user_id
# InvoiceDate -> datestamp
# Quantity -> purchased
# revenue -> paidamount
# StockCode -> item_id
event_df = raw_df.rename(columns={
    "CustomerID": "user_id",
    "InvoiceDate": "datestamp",
    "Quantity": "purchased",
    "revenue": "paidamount",
    "StockCode": "item_id"
}).copy()

# TODO:
# datestamp 컬럼을 datetime 형식으로 변환하세요.
event_df["datestamp"] = pd.to_datetime(event_df['datestamp'])

# 현재 데이터는 구매 로그이므로 실제 클릭 로그는 없습니다.
# 따라서 구매가 발생한 row는 반드시 클릭으로 간주하고,
# 구매가 없는 row에 대해서는 일부만 클릭으로 생성하여
# 퍼널 구조(impression → click → conversion)를 단순하게 구성합니다.
np.random.seed(42)

# 기본값 0
event_df["clicked"] = 0

# 구매 발생 row는 반드시 클릭
event_df.loc[event_df["purchased"] > 0, "clicked"] = 1

# 구매가 없는 row만 일부 클릭으로 설정
non_purchase_mask = event_df["purchased"] == 0
event_df.loc[non_purchase_mask, "clicked"] = np.random.binomial(
    1, 0.3, size=non_purchase_mask.sum()
)

# TODO:
# purchased가 1 이상이면 1, 아니면 0이 되도록
# converted(구매 발생 여부) 컬럼을 생성하세요.
event_df["converted"] = (event_df['purchased'] >= 1).astype(int)

# TODO:
# 아래 컬럼만 남겨 event_df를 정리하세요.
# user_id, datestamp, item_id, clicked, converted, purchased, paidamount
event_df = event_df[[
    'user_id', 'datestamp', 'item_id', 'clicked', 'converted', 'purchased', 'paidamount'
]]

print(event_df.head())
print(event_df.columns.tolist())

   user_id           datestamp item_id  clicked  converted  purchased  \
0    15311 2010-12-01 09:41:00   22139        1          1         23   
1    15311 2010-12-01 09:41:00   84854        1          1          5   
2    15311 2010-12-01 09:41:00   22411        1          1         10   
3    15311 2010-12-01 09:41:00   82567        1          1          2   
4    15311 2010-12-01 09:41:00   21672        1          1          6   

   paidamount  
0       97.75  
1       24.75  
2       19.50  
3        4.20  
4        7.50  
['user_id', 'datestamp', 'item_id', 'clicked', 'converted', 'purchased', 'paidamount']


In [5]:
# Step 3. Variant 생성
def split_user(user_id):
    h = hashlib.md5(str(user_id).encode())

    # TODO:
    # md5 해시값을 정수형으로 변환한 뒤
    # 홀수면 "test", 짝수면 "control"을 반환하세요.
    return "test" if int(h.hexdigest(), 16) % 2 == 1 else "control"


# TODO:
# user_id 기준 고유 사용자 목록으로 user_variant_df를 생성하세요.
user_variant_df = pd.DataFrame({
    "user_id": event_df['user_id'].unique()
})

# TODO:
# user_id에 split_user 함수를 적용하여 variant_id 컬럼을 생성하세요.
user_variant_df["variant_id"] = user_variant_df['user_id'].apply(split_user)

print(user_variant_df.head())
print(user_variant_df["variant_id"].value_counts())

   user_id variant_id
0    15311    control
1    16029       test
2    12431    control
3    13747       test
4    15513    control
variant_id
control    506
test       494
Name: count, dtype: int64


In [6]:
# Step 4. 사용자 메타데이터 생성
np.random.seed(42)

# user_id별 age, gender를 랜덤 생성한 user_metadata_df를 만듭니다.
# age: ["20s", "30s", "40s", "50s"]
# gender: ["M", "F"]
user_metadata_df = pd.DataFrame({
    "user_id": sorted(event_df["user_id"].dropna().unique()),
    "age": np.random.choice(["20s", "30s", "40s", "50s"], size=user_variant_df.shape[0]),
    "gender": np.random.choice(["M", "F"], size=user_variant_df.shape[0])
})

print(user_metadata_df.head())

   user_id  age gender
0    12346  40s      F
1    12348  50s      M
2    12352  20s      M
3    12359  40s      M
4    12360  40s      M


In [7]:
# Step 5. DuckDB 적재
con = duckdb.connect("ab_test_tableau.duckdb")

con.execute("CREATE SCHEMA IF NOT EXISTS raw_data;")
con.execute("CREATE SCHEMA IF NOT EXISTS analytics;")

con.register("tmp_event_df", event_df)
con.register("tmp_user_variant_df", user_variant_df)
con.register("tmp_user_metadata_df", user_metadata_df)

con.execute("DROP TABLE IF EXISTS raw_data.user_event;")
con.execute("DROP TABLE IF EXISTS raw_data.user_variant;")
con.execute("DROP TABLE IF EXISTS raw_data.user_metadata;")

con.execute("""
CREATE TABLE raw_data.user_event AS
SELECT * FROM tmp_event_df
""")

con.execute("""
CREATE TABLE raw_data.user_variant AS
SELECT * FROM tmp_user_variant_df
""")

con.execute("""
CREATE TABLE raw_data.user_metadata AS
SELECT * FROM tmp_user_metadata_df
""")

In [8]:
# Step 6. Base Table 생성
con.execute("DROP TABLE IF EXISTS analytics.ab_base;")

# TODO:
# analytics.ab_base를 생성하세요.
# event_date는 datestamp를 DATE로 변환한 값입니다.
# user_event, user_variant, user_metadata를 user_id 기준으로 결합하세요.
con.execute("""
CREATE TABLE analytics.ab_base AS
SELECT
    e.user_id,
    CAST(e.datestamp AS DATE) AS event_date,
    e.datestamp,
    e.item_id,
    e.clicked,
    e.converted,
    e.purchased,
    e.paidamount,
    v.variant_id,
    m.gender,
    m.age
FROM raw_data.user_event e
LEFT JOIN raw_data.user_variant v
    ON e.user_id = v.user_id
LEFT JOIN raw_data.user_metadata m
    ON e.user_id = m.user_id
""")

base_df = con.execute("""
SELECT *
FROM analytics.ab_base
LIMIT 10
""").df()

print(base_df)

   user_id event_date           datestamp item_id  clicked  converted  \
0    14502 2011-03-13 2011-03-13 12:39:00   22804        1          0   
1    17994 2011-06-03 2011-06-03 09:50:00   22693        1          0   
2    14057 2011-03-03 2011-03-03 14:56:00   23004        0          0   
3    14502 2011-04-18 2011-04-18 17:29:00  85049C        1          0   
4    18232 2011-07-05 2011-07-05 11:10:00   23085        0          0   
5    14400 2011-07-13 2011-07-13 15:50:00   21900        0          0   
6    17841 2011-06-21 2011-06-21 15:41:00   21790        1          0   
7    16066 2011-10-09 2011-10-09 11:53:00   23372        0          0   
8    17757 2011-07-10 2011-07-10 15:38:00   22439        0          0   
9    15910 2011-12-09 2011-12-09 10:51:00   23103        0          0   

   purchased  paidamount variant_id gender  age  
0          0         0.0       test      F  20s  
1          0         0.0    control      M  20s  
2          0         0.0    control      M  40

In [9]:
# Step 7. Daily Cube 생성
con.execute("DROP TABLE IF EXISTS analytics.ab_cube_daily;")

# TODO:
# analytics.ab_cube_daily를 생성하세요.
# event_date, variant_id, gender, age 기준으로 그룹화하고
# 아래 지표를 집계하세요.
# - impressions: COUNT(*)
# - users: COUNT(DISTINCT user_id)
# - clicks: SUM(clicked)
# - conversions: SUM(converted)
# - purchases: SUM(purchased)
# - revenue: SUM(paidamount)
con.execute("""
CREATE TABLE analytics.ab_cube_daily AS
SELECT
    event_date,
    variant_id,
    COALESCE(gender, 'unknown') AS gender,
    COALESCE(age, 'unknown') AS age,
    COUNT(*) AS impressions,
    COUNT(DISTINCT user_id) AS users,
    SUM(clicked) AS clicks,
    SUM(converted) AS conversions,
    SUM(purchased) AS purchases,
    SUM(paidamount) AS revenue
FROM analytics.ab_base
GROUP BY 1, 2, 3, 4
""")

cube_df = con.execute("""
SELECT *
FROM analytics.ab_cube_daily
ORDER BY event_date, variant_id, gender, age
LIMIT 20
""").df()

print(cube_df)

   event_date variant_id gender  age  impressions  users  clicks  conversions  \
0  2010-12-01    control      F  30s           86      1    36.0         20.0   
1  2010-12-01    control      F  50s          185      3    92.0         48.0   
2  2010-12-01    control      M  20s          442      2   213.0        116.0   
3  2010-12-01    control      M  30s           85      1    43.0         23.0   
4  2010-12-01    control      M  50s          239      4   124.0         61.0   
5  2010-12-01       test      F  20s            4      1     1.0          1.0   
6  2010-12-01       test      F  30s          346      3   154.0         79.0   
7  2010-12-01       test      F  40s           90      2    48.0         25.0   
8  2010-12-01       test      F  50s          370      3   175.0         86.0   
9  2010-12-01       test      M  30s            4      1     2.0          1.0   
10 2010-12-01       test      M  40s           42      3    22.0         14.0   
11 2010-12-01       test    

In [10]:
# Step 8. CSV 저장
full_base_df = con.execute("SELECT * FROM analytics.ab_base").df()
full_cube_df = con.execute("SELECT * FROM analytics.ab_cube_daily").df()

# TODO:
# Base Table과 Daily Cube를 각각 CSV 파일로 저장하세요.
full_base_df.to_csv("ab_base_analysis.csv", index=False)
full_cube_df.to_csv("ab_daily_cube.csv", index=False)